# Griffiths Problem 1.10 — Vectors, Pseudovectors, and Pseudoscalars

Parts (a)–(d): translation, inversion, cross products, scalar triple products.  
Stack: **SymPy** (analytic) · **NumPy** (numerical) · **Matplotlib** (3-D viz) · **PyTorch** (batch tensor algebra)

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
import torch

sp.init_printing(use_latex='mathjax')   # renders SymPy exprs as LaTeX in notebook
plt.rcParams.update({'font.size': 13, 'figure.dpi': 110})
print('SymPy', sp.__version__, '| NumPy', np.__version__, '| Torch', torch.__version__)

---
## SymPy Loop — All Four Transformation Types at Once

Loop over every combination of (object type, transformation) and let SymPy
compute the ratio $\bar{Q}/Q$ symbolically.  
`init_printing` renders each result as LaTeX inline.

In [ ]:
from IPython.display import display, Math
import sympy as sp

sp.init_printing(use_latex='mathjax')

# ── Shared symbols ──────────────────────────────────────────────────────
Ax,Ay,Az = sp.symbols('A_x A_y A_z', real=True)
Bx,By,Bz = sp.symbols('B_x B_y B_z', real=True)
Cx,Cy,Cz = sp.symbols('C_x C_y C_z', real=True)
a_s       = sp.Symbol('a', real=True)

A = sp.Matrix([Ax, Ay, Az])
B = sp.Matrix([Bx, By, Bz])
C = sp.Matrix([Cx, Cy, Cz])

P = sp.eye(3) * -1   # parity / inversion matrix

# ── Translation operator (shift y by a) ────────────────────────────────
def T(v):
    """Translate: tip and tail both shift by (0,a,0), so difference is unchanged."""
    shift = sp.Matrix([0, a_s, 0])
    return v   # vector = tip - tail, shift cancels

# ── Define the four quantity types ─────────────────────────────────────
#   Each entry: (label, latex_name, expr_fn, inv_expr_fn, expected_ratio)
#   expr_fn / inv_expr_fn return a SymPy scalar or Matrix

def polar_orig():       return A
def polar_inv():        return P * A

def pseudo_orig():      return A.cross(B)
def pseudo_inv():       return (P*A).cross(P*B)

def scalar_orig():      return A.dot(A)            # |A|² is simpler to ratio
def scalar_inv():       return (P*A).dot(P*A)

def pscalar_orig():     return A.dot(B.cross(C))
def pscalar_inv():      return (P*A).dot((P*B).cross(P*C))

cases = [
    ('Polar vector',    r'\mathbf{A}',
     polar_orig,   polar_inv,   -1, True),
    ('Pseudovector',    r'\mathbf{A}\times\mathbf{B}',
     pseudo_orig,  pseudo_inv,  +1, True),
    ('Scalar',          r'|\mathbf{A}|^2',
     scalar_orig,  scalar_inv,  +1, False),
    ('Pseudoscalar',    r'\mathbf{A}\cdot(\mathbf{B}\times\mathbf{C})',
     pscalar_orig, pscalar_inv, -1, False),
]

print('='*70)
print(f'  {"Type":18}  under translation    under inversion P')
print('='*70)

for name, latex, orig_fn, inv_fn, expected_sign, is_vector in cases:
    orig = orig_fn()
    inv  = inv_fn()

    # ── inversion ratio ──────────────────────────────────────────────
    if is_vector:
        diff = sp.simplify(inv - expected_sign * orig)
        ratio_str = f'+1·Q' if expected_sign == 1 else '-1·Q'
        check = '✓' if diff == sp.zeros(3,1) else '✗'
    else:
        ratio = sp.simplify(inv / orig)
        ratio_str = sp.latex(ratio)
        check = '✓' if ratio == expected_sign else '✗'

    # ── translation: vectors and scalars built from vectors are unchanged ─
    trans_result = 'unchanged'   # always true for all four types

    print(f'  {name:18}  {trans_result:20} {ratio_str:12} {check}')

    # ── display SymPy expression with init_printing LaTeX ────────────
    display(Math(
        rf'\text{{{name}}}: \quad'
        rf'Q = {latex}, \quad'
        rf'P\,Q = {sp.latex(inv)}'
    ))
    print()

print('='*70)

# ── Extra: loop over all components of pseudo and polar ────────────────
print('\nComponent-wise inversion ratios (SymPy loop):')
for comp_idx, comp_label in enumerate(['x', 'y', 'z']):
    polar_ratio = sp.simplify(polar_inv()[comp_idx] / polar_orig()[comp_idx])
    pseudo_ratio = sp.simplify(pseudo_inv()[comp_idx] / pseudo_orig()[comp_idx])
    display(Math(
        rf'\bar{{A}}_{{{comp_label}}} / A_{{{comp_label}}} = {sp.latex(polar_ratio)}'
        rf'\qquad'
        rf'(\bar{{A\times B}})_{{{comp_label}}} / (A\times B)_{{{comp_label}}} = {sp.latex(pseudo_ratio)}'
    ))

---
## Part (a) — Translation  $\bar{x}=x,\; \bar{y}=y-a,\; \bar{z}=z$

A **translation** shifts the *origin*, not the axes directions.  
Vector **components** are differences of coordinates, so the shift cancels:
$$\bar{A}_i = \bar{r}_i^{\,(\text{tip})} - \bar{r}_i^{\,(\text{tail})} = A_i \quad\forall\,i$$
Vectors are **translation-invariant**.

In [ ]:
# ── SymPy: symbolic proof ──────────────────────────────────────────────
a, x1, y1, z1, x2, y2, z2 = sp.symbols('a x1 y1 z1 x2 y2 z2', real=True)

# vector from point 1 to point 2 in original frame
A = sp.Matrix([x2 - x1, y2 - y1, z2 - z1])

# translated coordinates  (x̄=x, ȳ=y-a, z̄=z)
def translate(x, y, z):
    return x, y - a, z

tx1, ty1, tz1 = translate(x1, y1, z1)
tx2, ty2, tz2 = translate(x2, y2, z2)
A_bar = sp.Matrix([tx2 - tx1, ty2 - ty1, tz2 - tz1])

diff = sp.simplify(A_bar - A)
print('Ā − A =', diff.T, '  ← zero proves translation invariance')

# ── NumPy: loop over 200 random vectors ───────────────────────────────
rng = np.random.default_rng(0)
a_val = 3.7
tips  = rng.uniform(-5, 5, (200, 3))
tails = rng.uniform(-5, 5, (200, 3))
vecs  = tips - tails

tips_bar  = tips  - np.array([0, a_val, 0])
tails_bar = tails - np.array([0, a_val, 0])
vecs_bar  = tips_bar - tails_bar

max_err = np.max(np.abs(vecs_bar - vecs))
print(f'NumPy max |Ā−A| over 200 random vectors: {max_err:.2e}  (machine zero)')

# ── Matplotlib: show one vector before/after translation ──────────────
fig, ax = plt.subplots(figsize=(7, 4))
v = np.array([2.0, 1.5])
tail_orig = np.array([1.0, 2.0]);  tip_orig = tail_orig + v
tail_tr   = tail_orig - np.array([0, a_val/2])
tip_tr    = tail_tr + v

kw = dict(length_includes_head=True, head_width=0.15, head_length=0.1)
ax.arrow(*tail_orig, *v, color='royalblue', **kw, label='original')
ax.arrow(*tail_tr,   *v, color='tomato',    **kw, label='translated origin')
ax.axhline(2.0,  color='royalblue', ls='--', lw=0.8, alpha=0.5)
ax.axhline(2.0 - a_val/2, color='tomato', ls='--', lw=0.8, alpha=0.5)
ax.set_xlim(0, 5); ax.set_ylim(-0.5, 4.5)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Part (a): Translation — vector components unchanged')
ax.legend(); plt.tight_layout(); plt.show()

---
## Part (b) — Inversion  $\bar{x}=-x,\; \bar{y}=-y,\; \bar{z}=-z$

Inversion is a **parity transformation** $P$: every coordinate flips sign.  
A true (**polar**) vector transforms as $\bar{\mathbf{A}} = -\mathbf{A}$.  
Equivalently the Jacobian is $J = \det(-I) = -1$.

In [ ]:
# ── SymPy: parity matrix ───────────────────────────────────────────────
P = sp.Matrix([[-1,0,0],[0,-1,0],[0,0,-1]])
Ax, Ay, Az = sp.symbols('A_x A_y A_z', real=True)
A_sym = sp.Matrix([Ax, Ay, Az])
A_inv = P * A_sym
print('Under inversion P·A =', A_inv.T)
print('det(P) =', P.det())

# ── Torch: batch inversion of 1000 vectors ────────────────────────────
torch.manual_seed(42)
vecs_t = torch.randn(1000, 3)
P_t    = -torch.eye(3)
vecs_inv = (P_t @ vecs_t.T).T          # broadcast: (3,3)·(3,N)→(N,3)
assert torch.allclose(vecs_inv, -vecs_t), 'Inversion failed!'
print('Torch: P·v == -v  ✓  for all 1000 vectors')

# ── 3-D plot: vector before / after inversion ─────────────────────────
fig = plt.figure(figsize=(7, 6))
ax  = fig.add_subplot(111, projection='3d')

v0 = np.array([1.5, 1.0, 1.2])
ax.quiver(0,0,0, *v0,  color='royalblue', linewidth=2, arrow_length_ratio=0.15, label='A')
ax.quiver(0,0,0, *-v0, color='tomato',    linewidth=2, arrow_length_ratio=0.15, label='-A (inverted)')

for s, col in [(1,'royalblue'),(-1,'tomato')]:
    for axis in np.eye(3):
        ax.quiver(0,0,0, *(s*axis*0.5), color=col, linewidth=0.6, alpha=0.4, arrow_length_ratio=0.2)

ax.set_xlim(-2,2); ax.set_ylim(-2,2); ax.set_zlim(-2,2)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('Part (b): Inversion — polar vector flips sign')
ax.legend(); plt.tight_layout(); plt.show()

---
## Part (c) — Cross Product under Inversion: **Pseudovector**

Under inversion $\mathbf{A}\to-\mathbf{A}$ and $\mathbf{B}\to-\mathbf{B}$, so:
$$\overline{(\mathbf{A}\times\mathbf{B})} = (-\mathbf{A})\times(-\mathbf{B}) = +\mathbf{A}\times\mathbf{B}$$
The cross product does **not** flip sign — it is a **pseudovector** (axial vector).  
Classical examples: angular momentum $\mathbf{L}=\mathbf{r}\times\mathbf{p}$, magnetic field $\mathbf{B}$, torque $\boldsymbol{\tau}$.

In [ ]:
# ── SymPy: analytic verification ──────────────────────────────────────
Bx, By, Bz = sp.symbols('B_x B_y B_z', real=True)
B_sym = sp.Matrix([Bx, By, Bz])

cross_orig = A_sym.cross(B_sym)
cross_inv  = (P * A_sym).cross(P * B_sym)
diff_c = sp.simplify(cross_inv - cross_orig)
print('(P·A)×(P·B) − A×B =', diff_c.T, '  ← zero: pseudovector!')

# ── NumPy loop: test 5000 random pairs ────────────────────────────────
rng = np.random.default_rng(7)
A_n = rng.standard_normal((5000, 3))
B_n = rng.standard_normal((5000, 3))
cross_n      = np.cross(A_n,  B_n)
cross_n_inv  = np.cross(-A_n, -B_n)
max_err_c = np.max(np.abs(cross_n_inv - cross_n))
print(f'NumPy max |(−A)×(−B) − A×B| over 5000 pairs: {max_err_c:.2e}')

# ── Torch: same check as batch matrix op ──────────────────────────────
A_t = torch.tensor(A_n, dtype=torch.float64)
B_t = torch.tensor(B_n, dtype=torch.float64)
cross_t     = torch.linalg.cross(A_t,  B_t)
cross_t_inv = torch.linalg.cross(-A_t, -B_t)
err_t = (cross_t_inv - cross_t).abs().max().item()
print(f'Torch max err: {err_t:.2e}  ✓')

# ── 3-D plot: A, B, and A×B before and after inversion ────────────────
fig = plt.figure(figsize=(10, 5))
titles = ['Original frame', 'Inverted frame (P)']
signs  = [1, -1]

va = np.array([1.0, 0.2, 0.1])
vb = np.array([0.1, 1.0, 0.2])
vc = np.cross(va, vb)

for idx, (s, title) in enumerate(zip(signs, titles)):
    ax = fig.add_subplot(1, 2, idx+1, projection='3d')
    ax.quiver(0,0,0, *(s*va), color='royalblue', lw=2, arrow_length_ratio=0.15, label='A')
    ax.quiver(0,0,0, *(s*vb), color='seagreen',  lw=2, arrow_length_ratio=0.15, label='B')
    ax.quiver(0,0,0, *(vc),   color='crimson',   lw=2.5, arrow_length_ratio=0.15, label='A×B (same!)')
    ax.set_xlim(-1.5,1.5); ax.set_ylim(-1.5,1.5); ax.set_zlim(-1.5,1.5)
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
    ax.set_title(title); ax.legend(fontsize=9)

plt.suptitle('Part (c): Cross product = pseudovector — does NOT flip under inversion', fontsize=12)
plt.tight_layout(); plt.show()

### Classical pseudovector quantities

| Quantity | Expression | Type |
|----------|-----------|------|
| Angular momentum | $\mathbf{L}=\mathbf{r}\times\mathbf{p}$ | pseudovector |
| Magnetic field | $\mathbf{B}=\nabla\times\mathbf{A}$ | pseudovector |
| Torque | $\boldsymbol{\tau}=\mathbf{r}\times\mathbf{F}$ | pseudovector |
| Angular velocity | $\boldsymbol{\omega}$ (via $\mathbf{v}=\boldsymbol{\omega}\times\mathbf{r}$) | pseudovector |

In [ ]:
# ── Torch: verify L = r×p transforms as pseudovector ──────────────────
N = 2000
torch.manual_seed(0)
r = torch.randn(N, 3, dtype=torch.float64)
p = torch.randn(N, 3, dtype=torch.float64)

L_orig = torch.linalg.cross(r, p)
L_inv  = torch.linalg.cross(-r, -p)      # both r and p are polar → flip
print('L under P: max |L_inv − L_orig| =', (L_inv - L_orig).abs().max().item(), ' ← pseudovector ✓')

# cross of two pseudovectors is a polar vector
C1 = torch.linalg.cross(r, p)            # pseudovector
C2 = torch.linalg.cross(r, p) * 0.5     # pseudovector
polar_from_pseudo = torch.linalg.cross(C1, C2)
# under inversion: pseudo×pseudo = +×+ (no flip) → should stay same sign
check = torch.linalg.cross(-C1, -C2)     # pseudovectors don't flip
print('pseudo×pseudo (polar): max err =', (check - polar_from_pseudo).abs().max().item())

# loop over a table of examples
examples = [
    ('polar × polar',      True,  True,  'pseudovector'),
    ('pseudo × pseudo',    False, False, 'polar vector'),
    ('polar × pseudo',     True,  False, 'polar vector'),
]
print('\n Cross-product type table:')
print(f'{"Left":20} {"Right":20} {"Result":20}')
print('-'*62)
for left, l_flips, r_flips, result in examples:
    l, r_str = left.split(' × ')
    print(f'{l:20} {r_str:20} → {result}')

---
## Part (d) — Scalar Triple Product under Inversion: **Pseudoscalar**

$$\mathbf{A}\cdot(\mathbf{B}\times\mathbf{C}) = \det\begin{pmatrix}A_x&A_y&A_z\\B_x&B_y&B_z\\C_x&C_y&C_z\end{pmatrix}$$

Under inversion $\mathbf{A}\to-\mathbf{A}$, $\mathbf{B}\to-\mathbf{B}$, $\mathbf{C}\to-\mathbf{C}$:
$$\overline{\mathbf{A}\cdot(\mathbf{B}\times\mathbf{C})} = (-\mathbf{A})\cdot\bigl((-\mathbf{B})\times(-\mathbf{C})\bigr)
= (-\mathbf{A})\cdot(+\mathbf{B}\times\mathbf{C}) = -\mathbf{A}\cdot(\mathbf{B}\times\mathbf{C})$$

It **changes sign** under inversion → **pseudoscalar**.  
Geometrically: the signed volume of the parallelepiped flips when the frame is reflected.

In [ ]:
# ── SymPy: symbolic proof ──────────────────────────────────────────────
Cx, Cy, Cz = sp.symbols('C_x C_y C_z', real=True)
C_sym = sp.Matrix([Cx, Cy, Cz])

stp_orig = A_sym.dot(B_sym.cross(C_sym))
stp_inv  = (P*A_sym).dot((P*B_sym).cross(P*C_sym))
ratio    = sp.simplify(stp_inv / stp_orig)
print('stp_inv / stp_orig =', ratio, '  ← pseudoscalar flips sign!')

# ── NumPy loop: 8000 random triples ───────────────────────────────────
rng = np.random.default_rng(99)
A_n = rng.standard_normal((8000, 3))
B_n = rng.standard_normal((8000, 3))
C_n = rng.standard_normal((8000, 3))

stp  = np.einsum('ni,ni->n', A_n, np.cross(B_n, C_n))
stp_i = np.einsum('ni,ni->n', -A_n, np.cross(-B_n, -C_n))

max_err_stp = np.max(np.abs(stp_i + stp))   # should be near zero (stp_i = -stp)
print(f'NumPy max |stp_inv + stp| (should be ~0): {max_err_stp:.2e}')
print(f'NumPy: sign always flips: {np.all(np.sign(stp_i) == -np.sign(stp[np.abs(stp)>1e-12]))}')

# ── Torch: batch determinant = scalar triple product ──────────────────
A_t = torch.tensor(A_n[:1000], dtype=torch.float64)
B_t = torch.tensor(B_n[:1000], dtype=torch.float64)
C_t = torch.tensor(C_n[:1000], dtype=torch.float64)

M     = torch.stack([A_t, B_t, C_t], dim=1)     # (N,3,3)
M_inv = torch.stack([-A_t, -B_t, -C_t], dim=1)

det_orig = torch.linalg.det(M)
det_inv  = torch.linalg.det(M_inv)
err_det  = (det_inv + det_orig).abs().max().item()
print(f'Torch det(M_inv) + det(M) max err: {err_det:.2e}  ✓  pseudoscalar confirmed')

In [ ]:
# ── 3-D visualization: parallelepiped volume before/after inversion ───
def draw_parallelepiped(ax, A, B, C, color, alpha=0.15, label=''):
    """Draw faces of parallelepiped spanned by A, B, C from origin."""
    verts = np.array([[0,0,0], A, B, C,
                       A+B, A+C, B+C, A+B+C])
    faces = [
        [verts[0],verts[1],verts[4],verts[2]],
        [verts[0],verts[1],verts[5],verts[3]],
        [verts[0],verts[2],verts[6],verts[3]],
        [verts[7],verts[4],verts[2],verts[6]],
        [verts[7],verts[4],verts[1],verts[5]],
        [verts[7],verts[6],verts[3],verts[5]],
    ]
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection
    poly = Poly3DCollection(faces, alpha=alpha, facecolor=color, edgecolor=color, linewidth=0.5)
    ax.add_collection3d(poly)
    for v, lbl in zip([A,B,C], ['A','B','C']):
        ax.quiver(0,0,0,*v, color=color, lw=2, arrow_length_ratio=0.15)
        ax.text(*(v*1.05), lbl, fontsize=10, color=color)
    vol = float(np.dot(A, np.cross(B, C)))
    ax.set_title(f'{label}\nVol = A·(B×C) = {vol:.2f}', fontsize=11)

va = np.array([1.2, 0.3, 0.1])
vb = np.array([0.2, 1.0, 0.3])
vc = np.array([0.1, 0.2, 0.9])

fig = plt.figure(figsize=(11, 5))
for idx, (s, title) in enumerate([(1,'Original'), (-1,'Inverted (P)')]):
    ax = fig.add_subplot(1, 2, idx+1, projection='3d')
    draw_parallelepiped(ax, s*va, s*vb, s*vc,
                        color='royalblue' if s==1 else 'tomato',
                        label=title)
    lim = 1.8
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_zlim(-lim, lim)
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')

plt.suptitle('Part (d): Scalar triple product = pseudoscalar — sign flips under inversion', fontsize=12)
plt.tight_layout(); plt.show()

---
## Summary Table

| Object | Under translation | Under inversion $P$ | Example |
|--------|:-----------------:|:------------------:|--------|
| **Polar vector** | unchanged | $\bar{\mathbf{A}} = -\mathbf{A}$ | position $\mathbf{r}$, force $\mathbf{F}$, $\mathbf{E}$-field |
| **Pseudovector** (axial) | unchanged | $\bar{\mathbf{A}} = +\mathbf{A}$ | $\mathbf{L}$, $\mathbf{B}$, $\boldsymbol{\tau}$ |
| **Scalar** | unchanged | $\bar{s} = +s$ | mass, charge, $\|\mathbf{A}\|$ |
| **Pseudoscalar** | unchanged | $\bar{s} = -s$ | $\mathbf{A}\cdot(\mathbf{B}\times\mathbf{C})$, helicity |

In [ ]:
# ── Combined summary: inversion sign flip across all four types ────────
N = 300
rng = np.random.default_rng(5)

A_n = rng.standard_normal((N, 3))
B_n = rng.standard_normal((N, 3))
C_n = rng.standard_normal((N, 3))

polar    = A_n                            # polar vector, x-component
pseudo   = np.cross(A_n, B_n)            # pseudovector, x-component
scalar   = np.linalg.norm(A_n, axis=1)   # scalar (magnitude)
pscalar  = np.einsum('ni,ni->n', A_n, np.cross(B_n, C_n))  # pseudoscalar

polar_i   = -A_n
pseudo_i  = np.cross(-A_n, -B_n)
scalar_i  = np.linalg.norm(-A_n, axis=1)
pscalar_i = np.einsum('ni,ni->n', -A_n, np.cross(-B_n, -C_n))

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
labels_all = [
    ('Polar vector  $A_x$',       polar[:,0],   polar_i[:,0],   'royalblue'),
    ('Pseudovector  $(A×B)_x$',   pseudo[:,0],  pseudo_i[:,0],  'seagreen'),
    ('Scalar  $|A|$',             scalar,       scalar_i,       'darkorange'),
    ('Pseudoscalar  $A·(B×C)$',   pscalar,      pscalar_i,      'crimson'),
]
for ax, (title, orig, inv, col) in zip(axes.flat, labels_all):
    ax.scatter(orig, inv, s=6, color=col, alpha=0.5)
    lo = min(orig.min(), inv.min()); hi = max(orig.max(), inv.max())
    ax.plot([lo,hi],[lo,hi],  'k--', lw=0.8, label='y=x (no change)')
    ax.plot([lo,hi],[-hi,-lo],'r--', lw=0.8, label='y=−x (flip)')
    ax.set_xlabel('original'); ax.set_ylabel('after inversion')
    ax.set_title(title); ax.legend(fontsize=8)

plt.suptitle('How each object type transforms under parity inversion $P$', fontsize=13)
plt.tight_layout(); plt.show()

# ── Final Torch loop: verify all four types at once ───────────────────
torch.manual_seed(11)
A_t = torch.randn(5000, 3, dtype=torch.float64)
B_t = torch.randn(5000, 3, dtype=torch.float64)
C_t = torch.randn(5000, 3, dtype=torch.float64)

results = {
    'polar':     (A_t[:,0],  (-A_t)[:,0],  -1),
    'pseudo':    (torch.linalg.cross(A_t,B_t)[:,0],
                  torch.linalg.cross(-A_t,-B_t)[:,0], +1),
    'scalar':    (A_t.norm(dim=1), (-A_t).norm(dim=1), +1),
    'pscalar':   (torch.einsum('ni,ni->n', A_t, torch.linalg.cross(B_t,C_t)),
                  torch.einsum('ni,ni->n',-A_t, torch.linalg.cross(-B_t,-C_t)), -1),
}

print('\nTorch verification (5000 random samples each):')
print(f'{"type":12}  {"expected sign":15}  {"max |inv − expected|":22}  pass?')
print('-'*65)
for name, (orig, inv, sign) in results.items():
    expected = sign * orig
    err = (inv - expected).abs().max().item()
    print(f'{name:12}  {sign:+d} × original      {err:22.2e}  {"✓" if err < 1e-9 else "✗"}')